My solution for the titanic ml challenge

First I load training data from csv to data frame

In [93]:
import pandas as pd

df = pd.read_csv("./titanic/train.csv")

#df.shape
display(df.head(5))

#print("Survived females: ", ((df["Sex"] == "female") & (df["Survived"] == 1)).sum())
#print("Survived males: ", ((df["Sex"] == "male") & (df["Survived"] == 1)).sum())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Some data exploration, seeing how classes affected survivability and who exactly survived

In [94]:
data = []
for pclass in sorted(df["Pclass"].unique()):
    total = (df["Pclass"] == pclass).sum()
    total_males = ((df["Sex"] == "male") & (df["Pclass"] == pclass)).sum()
    total_females = ((df["Sex"] == "female") & (df["Pclass"] == pclass)).sum()

    survived = ((df["Pclass"] == pclass) & (df["Survived"] == 1))
    men_survived = ((df["Sex"] == "male") & survived).sum()
    female_survived = ((df["Sex"] == "female") & survived).sum()

    kids = ((df["Age"] < 18) & survived).sum()

    data.append({
        "Class": pclass,
        "Passengers": total,
        "Survived": survived.sum(),
        "Survived children": kids, 
        "Percentage (%)": survived.sum() / total * 100,
        "Survived men": men_survived,
        "Total men": total_males,
        "Survived females": female_survived,
        "Total Females": total_females
    })

summary = pd.DataFrame(data)
display(summary.style
        .format({"Percentage (%)": "{:.2f}%"})
        .hide(axis="index"))

Class,Passengers,Survived,Survived children,Percentage (%),Survived men,Total men,Survived females,Total Females
1,216,136,11,62.96%,45,122,91,94
2,184,87,21,47.28%,17,108,70,76
3,491,119,29,24.24%,47,347,72,144


Missing values

In [95]:
#df.shape

df.info()

print(df["Cabin"].isna().sum())
print(df["Age"].isna().sum())
print(df["Embarked"].isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
687
177
2


Missing values:
- cabin (687)
- age (177)
- Embarked (2)

I will skip Cabin column entirely for now, in future i will try to make it known/unknown and see if model improves

For age i will imputate the missing values with median age and also test model with age - known/unknown feature

Embarked - I will just fill this with the most popular port of embarkation present in dataset

Data will be filled based on training groups

Splitting data into training and testing split

In [96]:
from typing import cast

from pandas import DataFrame, Series
from sklearn.model_selection import train_test_split


X = df.drop(columns=["Cabin", "Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = cast(
    tuple[DataFrame, DataFrame, Series, Series],
    train_test_split(X, y, test_size=0.2, random_state=42, stratify=y),
)

print(X_train.shape)
print(X_test.shape)

(712, 10)
(179, 10)


Baseline - dummy classifier with most_frequent strategy so it will just spit the most frequent class in our dataset as its prediction. This is my baseline model which i will compare to my Logistic Regression

In [97]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

print("Classes:", dummy.classes_)
print("Class distribution:")
print(y_train.value_counts(normalize=True))

y_train_pred = dummy.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred, zero_division=0)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall : ", recall)
print("F1: ", f1)

Classes: [0 1]
Class distribution:
Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64
Accuracy:  0.6165730337078652
Precision:  0.0
Recall :  0.0
F1:  0.0


Cross validation to see how our dummy performs on different data

In [98]:
from sklearn.model_selection import cross_val_score


scores = cross_val_score(dummy, X_train, y_train, cv=5, scoring="accuracy")
print("CV accuracy scores: ", scores)
print("CV mean accuracy: ", scores.mean())

CV accuracy scores:  [0.61538462 0.61538462 0.61267606 0.61971831 0.61971831]
CV mean accuracy:  0.6165763813651137


Now I want to classify my features into numeric and categorical features because I want to preprocess them first to fill blank values 

In [99]:
numeric_features = ["Age", "Fare", "SibSp", "Parch"]

categorical_features = ["Pclass", "Sex", "Embarked"]

features = numeric_features + categorical_features

X_train_selected = X_train[features]

#X_train_selected.shape
X_train_selected.head(5)

,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked
692,NaN,56.4958,0,0,3,male,S
481,NaN,0.0000,0,0,2,male,S
527,NaN,221.7792,0,0,1,male,S
855,18.0,9.3500,0,1,3,female,S
801,31.0,26.2500,1,1,2,female,S


Now i want to build a pipeline which will transform my data into desired format.

In [100]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

Now  I can create my model using my data transformer pipeline

In [101]:
from sklearn.linear_model import LogisticRegression


model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

Lets train it!

In [102]:
logistic_scores = cross_val_score(model, X_train_selected, y_train, cv=5, scoring="accuracy")

print(logistic_scores)
print("Mean accuracy: ", logistic_scores.mean())

[0.7972028  0.76223776 0.8028169  0.80985915 0.81690141]
Mean accuracy:  0.7978036048458584


#### Model comparison with cross validation

Comparing DummyClassifier, Logistic Regression, Decision Tree and Random Forest using the same folds, preprocessing and input features.

In [103]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Defining our data split strategy, data will be shuffled before splitted into 5 folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
  "DummyClassifier": dummy,
  "LogisticRegression": LogisticRegression(max_iter=1000),
  "DecisionTree": DecisionTreeClassifier(
    max_depth=3,
    random_state=42
  ),
  "RandomForest": RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
  )
}

Now i write a loop which will compare different classificators

In [104]:
from sklearn.metrics import make_scorer


results = []

for name, classifier in models.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    cv_scores = cross_validate(
        model_pipeline,
        X_train_selected,
        y_train,
        cv=cv,
        scoring= {
            "accuracy": "accuracy",
            "precision": make_scorer(
                precision_score,
                zero_division=0
            ),
            "recall": "recall",
            "f1": "f1" 
        }
    )

    results.append({
        "Model": name,
        "Accuracy": cv_scores["test_accuracy"].mean(),
        "Precision": cv_scores["test_precision"].mean(),
        "Recall": cv_scores["test_recall"].mean(),
        "F1": cv_scores["test_f1"].mean()
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1
3,RandomForest,0.8160,0.8177,0.6700,0.7355
2,DecisionTree,0.8062,0.8142,0.6484,0.7180
1,LogisticRegression,0.8020,0.7652,0.6997,0.7301
0,DummyClassifier,0.6166,0.0000,0.0000,0.0000


#### Random forest Hyperparemeter tuning

In [105]:
from sklearn.model_selection import GridSearchCV

# Pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Our options we want to test
param_grid = {
    "classifier__max_depth": [3, 5, 7, None],
    "classifier__min_samples_leaf": [1, 3, 5]
}

# GridSearchCV will try all combinations of options and get the one with best accuracy
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_selected, y_train)

print("Best parameters: ", grid_search.best_params_)
print("Best CV accuracy: ", grid_search.best_score_)

results = pd.DataFrame(grid_search.cv_results_)

display(
    results[[
        "param_classifier__max_depth",
        "param_classifier__min_samples_leaf",
        "mean_test_score",
        "mean_train_score",
        "std_test_score",
        "rank_test_score"
    ]].sort_values("rank_test_score")
)

Best parameters:  {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 1}
Best CV accuracy:  0.821619225844578


,param_classifier__max_depth,param_classifier__min_samples_leaf,mean_test_score,mean_train_score,std_test_score,rank_test_score
6,7,1,0.821619,0.908711,0.030383,1
10,None,3,0.817414,0.894317,0.025115,2
4,5,3,0.815985,0.855336,0.024683,3
3,5,1,0.815976,0.862361,0.022673,4
0,3,1,0.814656,0.819172,0.032225,5
5,5,5,0.814577,0.850772,0.026343,6
8,7,5,0.813188,0.869383,0.027352,7
7,7,3,0.811799,0.876758,0.031533,8
1,3,3,0.810450,0.816714,0.029424,9
11,None,5,0.807594,0.873600,0.023678,10


#### Making predictions on test data with the best model

In [106]:
from sklearn.metrics import confusion_matrix


best_model = grid_search.best_estimator_

X_test_selected = X_test[features]

y_test_pred = best_model.predict(X_test_selected)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print("Test accuracy: ", accuracy)
print("Test precision: ", precision)
print("Test recall: ", recall)
print("Test F1: ", f1)

print(confusion_matrix(y_test, y_test_pred))


Test accuracy:  0.7988826815642458
Test precision:  0.8367346938775511
Test recall:  0.5942028985507246
Test F1:  0.6949152542372882
[[102   8]
 [ 28  41]]


#### Now lets train final model which will be trained on full data and it will be used for my kaggle titanic submission

In [107]:
from sklearn.base import clone

final_model = clone(grid_search.best_estimator_)

X_full = df[features]
y_full = df["Survived"]

final_model.fit(X_full, y_full)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['Age','Fare','SibSp',...,'Pclass','Sex','Embarked']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all rema

Load test data from kaggle

In [108]:
kaggle_test = pd.read_csv("titanic/test.csv")

display(kaggle_test)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


Make prediction and create submission.csv

In [109]:
X_kaggle = kaggle_test[features]

y_kaggle_pred = final_model.predict(X_kaggle)

submission = pd.DataFrame({
    "PassengerId": kaggle_test["PassengerId"],
    "Survived": y_kaggle_pred
})

print(submission.shape)
print(submission.isna().sum())
display(submission.head())

submission.to_csv("./titanic/submission.csv", index=False)

(418, 2)
PassengerId    0
Survived       0
dtype: int64


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,0


#### Feature engineering

Now i will try to improve my score using feature-engineering which means i will try to find new features in my dataset that could improve my model accuracy

In [110]:
X_train_FE = X_train_selected.copy()

X_train_FE["FamilySize"] = (
    X_train_FE["SibSp"] + X_train_FE["Parch"] + 1
)

X_train_FE["IsAlone"] = (
    (X_train_FE["FamilySize"] == 1).astype(int)
)

X_train_FE.head(10)

,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked,FamilySize,IsAlone
692,NaN,56.4958,0,0,3,male,S,1,1
481,NaN,0.0000,0,0,2,male,S,1,1
527,NaN,221.7792,0,0,1,male,S,1,1
855,18.0,9.3500,0,1,3,female,S,2,0
801,31.0,26.2500,1,1,2,female,S,3,0
652,21.0,8.4333,0,0,3,male,S,1,1
509,26.0,56.4958,0,0,3,male,S,1,1
557,NaN,227.5250,0,0,1,male,C,1,1
828,NaN,7.7500,0,0,3,male,Q,1,1
18,31.0,18.0000,1,0,3,female,S,2,0


Lets add our new features to our numeric_features

In [111]:
numeric_features_fe = numeric_features + ["FamilySize", "IsAlone"]

Now we build a new ColumnTransformer for our updated numeric features

In [112]:
preprocessor_fe = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features_fe),
    ("categorical", categorical_transformer, categorical_features)
])

Creating new model pipeline and training it using cross validation

In [113]:
model_fe = Pipeline([
    ("preprocessor", preprocessor_fe),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        min_samples_leaf=1,
        random_state=42
    ))
])

scores_fe = cross_validate(
    model_fe,
    X_train_FE,
    y_train,
    cv=cv,
    scoring= {
        "accuracy": "accuracy",
        "precision": make_scorer(
            precision_score,
            zero_division=0
        ),
        "recall": "recall",
        "f1": "f1" 
    }
)

scores_fe_df = pd.DataFrame(scores_fe)

mean_scores_fe = scores_fe_df.mean()
scores_fe_df.loc["Mean"] = mean_scores_fe

display(scores_fe_df.round(4))

print(scores_fe_df["test_accuracy"].mean())

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.1028,0.0163,0.8252,0.7885,0.7455,0.7664
1,0.0925,0.0158,0.7902,0.7551,0.6727,0.7115
2,0.0916,0.0158,0.8451,0.9231,0.6545,0.7660
3,0.0920,0.0160,0.7817,0.7255,0.6852,0.7048
4,0.0936,0.0159,0.8169,0.8500,0.6296,0.7234
Mean,0.0945,0.0160,0.8118,0.8084,0.6775,0.7344


0.8118093174431203


Well the accuracy didnt improve, lets try different features

#### New Feature - Title

Also for now we only test Title without the new IsAlone and FamilySize

In [114]:
X_train_title = X_train_selected.copy()

# Extracting title from Name
X_train_title["Title"] = (
    X_train["Name"].str.extract(r",\s*([^.]*)\.", expand=False)
)

X_train_title["Title"].value_counts()

Title
Mr          412
Miss        141
Mrs         107
Master       31
Dr            6
Rev           5
Col           2
Mlle          2
Major         1
Lady          1
Sir           1
Ms            1
Jonkheer      1
Don           1
Name: count, dtype: int64

Adding new feature and building new pipeline for it

In [115]:
categorical_features_title = categorical_features + ["Title"]

preprocessor_title = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features_title)
])

model_title = Pipeline([
    ("preprocessor", preprocessor_title),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        min_samples_leaf=1,
        random_state=42
    ))
])

scores_title = cross_validate(
    model_title,
    X_train_title,
    y_train,
    cv=cv,
    scoring= {
        "accuracy": "accuracy",
        "precision": make_scorer(
            precision_score,
            zero_division=0
        ),
        "recall": "recall",
        "f1": "f1" 
    }
)

scores_title_df = pd.DataFrame(scores_title)

mean_scores_title = scores_title_df.mean()
scores_title_df.loc["Mean"] = mean_scores_title

display(scores_title_df.round(4))

print(scores_title_df["test_accuracy"].mean())

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.1152,0.0163,0.8182,0.7843,0.7273,0.7547
1,0.0970,0.0168,0.8042,0.7872,0.6727,0.7255
2,0.0979,0.0160,0.8521,0.8696,0.7273,0.7921
3,0.0966,0.0167,0.8099,0.7755,0.7037,0.7379
4,0.1015,0.0170,0.8028,0.8095,0.6296,0.7083
Mean,0.1016,0.0166,0.8174,0.8052,0.6921,0.7437


0.8174332709543978


#### Grouping Rare Titles
Because we have many titles and some of them appear only once or twice, the model can wrongly assume these rare titles correlate with survivability so we choose the 4 most common titles and the rest will be put to into a fifth group called "Other"

In [119]:
X_train_title_grouped = X_train_title.copy()

common_titles = ["Mr", "Miss", "Mrs", "Master"]

X_train_title_grouped["Title"] = X_train_title_grouped["Title"].where(X_train_title_grouped["Title"].isin(common_titles), "Other")

print(X_train_title_grouped["Title"].value_counts())

Title
Mr        412
Miss      141
Mrs       107
Master     31
Other      21
Name: count, dtype: int64


Now we train our model on new grouped titles

In [122]:
scores_title_grouped = cross_validate(
    model_title,
    X_train_title_grouped,
    y_train,
    cv=cv,
    scoring= {
        "accuracy": "accuracy",
        "precision": make_scorer(
            precision_score,
            zero_division=0
        ),
        "recall": "recall",
        "f1": "f1" 
    }
)

scores_title_grouped_df = pd.DataFrame(scores_title_grouped)

mean_scores_title_grouped = scores_title_grouped_df.mean()
scores_title_grouped_df.loc["Mean"] = mean_scores_title_grouped

display(scores_title_grouped_df.round(4))

print(scores_title_grouped_df["test_accuracy"].mean())

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.1051,0.0168,0.8392,0.8333,0.7273,0.7767
1,0.0963,0.0167,0.7692,0.7292,0.6364,0.6796
2,0.0932,0.0164,0.8521,0.8864,0.7091,0.7879
3,0.0918,0.0158,0.8099,0.7872,0.6852,0.7327
4,0.0935,0.0159,0.7958,0.8049,0.6111,0.6947
Mean,0.0960,0.0163,0.8132,0.8082,0.6738,0.7343


0.8132276174529697


#### Another feature - IsAgeMissing

In [123]:
X_train_age = X_train_selected.copy()

X_train_age["AgeMissing"] = (
    X_train_age["Age"].isna().astype(int)
)

X_train_age["AgeMissing"].value_counts()

AgeMissing
0    575
1    137
Name: count, dtype: int64

New pipeline for new feature

In [124]:
numeric_features_age = numeric_features + ["AgeMissing"]

preprocessor_age = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features_age),
    ("categorical", categorical_transformer, categorical_features)
])

model_age = Pipeline([
    ("preprocessor", preprocessor_age),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        max_depth=7,
        min_samples_leaf=1,
        random_state=42
    ))
])

scores_age = cross_validate(
    model_age,
    X_train_age,
    y_train,
    cv=cv,
    scoring= {
        "accuracy": "accuracy",
        "precision": make_scorer(
            precision_score,
            zero_division=0
        ),
        "recall": "recall",
        "f1": "f1" 
    }
)

scores_age_df = pd.DataFrame(scores_age)

mean_scores_age = scores_age_df.mean()
scores_age_df.loc["Mean"] = mean_scores_age

display(scores_age_df.round(4))

print(mean_scores_age["test_accuracy"])

,fit_time,score_time,test_accuracy,test_precision,test_recall,test_f1
0,0.1195,0.0165,0.8462,0.8511,0.7273,0.7843
1,0.0981,0.0165,0.7832,0.7609,0.6364,0.6931
2,0.1074,0.2091,0.8451,0.9024,0.6727,0.7708
3,0.1033,0.0192,0.7676,0.7143,0.6481,0.6796
4,0.1039,0.0163,0.8028,0.8095,0.6296,0.7083
Mean,0.1064,0.0555,0.8090,0.8076,0.6628,0.7272


0.8089727174234216
